## Introduction

This notebook is broken up into sections by object type.

Each section contains several code blocks:

1. Request properties and export to `data/<object_type>_props.json`.
   - This was used to validate property names against the ones shown in the [Cal-ITP: Hubspot CRM Properties](https://docs.google.com/spreadsheets/d/1ZvU4WWS2K1QEtRIk16M83Xnv8X3-wPTB9Mkci_ZLyBE/edit) sheet, or to discover the available properties for the activity objects.
   - Uses the `get_object_props()` function defined in the Start block.
2. Define the properties we want to export.
   - A list of unused properties is also defined (based on prior manual observations) for the purpose of marking them in the exported summary sheet.
   - Must be run for subsequent block to work.
3. Generate summary CSV for the properties of this type.
   - Uses the `get_object_prop_summary()` function defined in the Start block.

### Setup

Requires a HubSpot [legacy private app](https://developers.hubspot.com/docs/apps/legacy-apps/private-apps/overview) with every available scope ending in `.read`.

The app will provide an access token that should be stored in an environment variable called `HUBSPOT_ACCESS_TOKEN`.

You can copy the sample environment file to get started; run the following command from the root of this repository:

```bash
cp .env.sample .env
```

Then open `.env` and fill in with your access token.

**The block below should be run before attempting any other blocks in this notebook.**
It will need to be re-run after restarting the kernel, as well.

In [ ]:
import os
from pathlib import Path

from hubspot import HubSpot

import pandas as pd

from data.utils import hubspot_to_df, write_csv_records, write_json_records


ACCESS_TOKEN = os.environ["HUBSPOT_ACCESS_TOKEN"]
ASSOCIATION_TYPES = ["contacts", "calls", "emails", "meetings", "notes", "tasks", "tickets"]

hubspot = HubSpot(access_token=ACCESS_TOKEN)


class RunMode:
    FULL = 0
    DRY = 1
    LIVE = 2


RUN_MODE = int(os.environ.get("HUBSPOT_API_RUN_MODE", RunMode.DRY))
print(f"Run mode: {RUN_MODE}")

if RUN_MODE == RunMode.FULL:
    # Delete existing data files
    [f.unlink() for f in Path("data").iterdir()]


def get_object_props(object_type: str, nicename: str = ""):
    """Exports details for all properties of an object type to a JSON file."""
    props = hubspot.crm.properties.core_api.get_all(object_type=object_type, archived=False)
    props_df = hubspot_to_df(props)
    if type(props_df) is pd.DataFrame:
        write_json_records(props_df, f"{nicename or object_type}_props.json")


def get_object_prop_summary(object_type: str, wanted_props: list[str], unused_props: list[str] = [], exclude_unused: bool = False, nicename: str = ""):
    """Exports sumary for given properties of an object type to a CSV file."""

    wanted_data = ["name", "label", "description", "type", "options", "hubspot_defined"]

    props = hubspot.crm.properties.core_api.get_all(object_type=object_type, archived=False)
    props_df = hubspot_to_df(props)

    if type(props_df) is pd.DataFrame:
        # filter df to drop columns not in wanted_data
        props_df = props_df[wanted_data]
        # filter df to drop rows not in wanted_props
        props_df = props_df[props_df.name.isin(wanted_props)]
        # filter out unused props, if requested, otherwise add column to indicate it
        if exclude_unused:
            props_df = props_df[~props_df.name.isin(unused_props)]
        else:
            props_df["unused"] = props_df["name"].apply(lambda row_name: row_name in unused_props)

        # simplify options object to just a list of values
        props_df["options"] = props_df["options"].apply(lambda row_options: "\n".join([option["label"] for option in row_options]))

        write_csv_records(props_df, f"{nicename or object_type}_prop_summary.csv")

## Contacts

In [ ]:
# request properties and export to data/contact_props.json
if RUN_MODE == RunMode.FULL:
    get_object_props("contact")

In [ ]:
# define list of properties that we want to export
contact_wanted_props = [
    "firstname",
    "lastname",
    "email",
    "phone",
    "jobtitle",
    "job_role",
    "notes_last_contacted",
    "district_rep",
    "conferences",
    "hubspot_owner_id",
    "account_manager",
    "address",
    "asset_tag",
    "benefits_monthly_product_subscription",
    "caltrans_districts",
    "city",
    "engagements_last_meeting_booked",
    "hs_marketable_status",
    "hs_object_id",
    "ntd_monthly_ridership_url",
]
contact_unused_props = [
    # consider dropping asset_tag? see comment in current sheet
]

In [ ]:
# export summary of important properties to contact_prop_summary.csv
get_object_prop_summary("contact", contact_wanted_props, contact_unused_props)

## Companies

In [ ]:
# request properties and export to data/company_props.json
if RUN_MODE == RunMode.FULL:
    get_object_props("company")

In [ ]:
# define list of properties that we want to export
company_wanted_props = [
    "name",
    "domain",
    "address",
    "email",
    "company_type",
    "caltrans_district",
    "airtable_record_id",
    "about_us",
    "account_manager",
    "additional_url",
    "agency_acronym",
    "agency_collect_signage_info_from_riders_additional_detail",
    "agency_collects_info_from_riders_regarding_signage_and_amenities",
    "agency_logo",
    "agency_name",
    "agency_priority",
    "agency_scheduling_challenges",
    "am_support_remix",
    "applicable_fare_discounts",
    "associated_transit_organizations",
    "benefits_form_notes",
    "benefits_marketing_assistance",
    "benefits_onboarding_call",
    "benefits_testing_assistance",
    "board_approval",
    "bus_stop_signs_and_amenities",
    "ca_assembly_district",
    "ca_senate_district",
    "cad_avl_vendor",
    "cal_itp_services",
    "cellular_provider",
    "city",
    "contract_for",
    "contract_url",
    "contracts_received",
    "country",
    "county",
    "createdate",
    "customer_journey_position",
    "customer_testimonial",
    "date_remix_application_received",
    "description_of_caps",
    "desired_launch_timeline",
    "discount_fare_caps",
    "discount_fare_description",
    "discount_fare_structure",
    "discount_group_fare_differences",
    "engagements_last_meeting_booked",
    "engagements_last_meeting_booked_medium",
    "existing_remix_customer",
    "factors_preventing_remix_use",
    "fare_calculation_software",
    "fare_collection_system",
    "fare_validator_vendor",
    "farebox_vendor",
    "first_contact_createdate",
    "first_deal_created_date",
    "flag_for_quarterly_tier_review",
    "fleet_details",
    "fleet_size",
    "google_maps_data",
    "gtfs_vendor",
    "help_with_product_creation",
    "hs_object_id",
    "hubspot_owner_id",
    "itp_id",
    "no_remix_training_reason",
    "notes_last_contacted",
    "notes_last_updated",
    "notes_next_activity_date",
    "ntd_id",
    "ntd_monthly_ridership_url",
    "number_of_stops_on_the_state_highway_network__shn_",
    "organization_type",
    "pathways_to_be_enabled_for_benefits",
    "payment_processor",
    "payment_processor_vendors",
    "permission_to_use_agency_logo_on_websites_",
    "phone",
    "physical_signage_priority_ranking",
    "produce_gtfs_data",
    "product_creation",
    "real_time_digital_signage_priority",
    "recent_deal_close_date",
    "remix_greatest_challenges",
    "remix_hopes",
    "remix_training_yes_no",
    "rider_benches_needs_met",
    "riders_physical_signage_needs_met",
    "riders_real_time_digital_signage_needs_met",
    "riders_shelter_needs_met",
    "route_details",
    "router_onboard",
    "services",
    "shelter_priority_ranking",
    "special_projects",
    "state",
    "tap_agency",
    "telecom_provider",
    "tier",
    "transit_processor",
    "vendor_type",
    "voms",
    "zip",
    "grants_eligibility",
    "apc_vendor",
    "asset_management_vendor",
    "cad_vendor",
    "data_management_and_analytics_vendor",
    "digital_signage_and_information_display_systems_vendor",
    "fleet_management_vendor",
    "funding_sources",
    "funding_sources__other_",
    "gtfs_status",
    "gtfs_use_cases",
    "interest_in_cal_itp_products",
    "next_schedule_change",
    "passenger_information_systems_vendor",
    "predictive_maintenance_vendor",
    "remix_agreement",
    "scheduling_changes",
    "scheduling_vendor",
    "scheduling_with_zeb",
    "technology_assistances",
    "technology_procurement",
    "voms__scheduling_",
    "days_to_close",
]
company_unused_props = [
    "benefits_form_notes",
    "benefits_onboarding_call",
    "date_remix_application_received",
    # to consider, see comments in current sheet:
    # agency_collect_signage_info_from_riders_additional_detail,
    # agency_collects_info_from_riders_regarding_signage_and_amenities,
    # agency_scheduling_challenges,
    # flag_for_quarterly_tier_review
]

In [ ]:
# export summary of important properties to company_prop_summary.csv
get_object_prop_summary("company", company_wanted_props, company_unused_props)

## Tickets

In [ ]:
# request properties and export to data/ticket_props.json
# - HAD TO ADD `tickets` SCOPE TO LEGACY APP
if RUN_MODE == RunMode.FULL:
    get_object_props("ticket")

In [ ]:
# define list of properties that we want to export
ticket_wanted_props = [
    "subject",
    "hs_pipeline",
    "hs_pipeline_stage",
    "hs_last_closed_date",
    "active_campaign",
    "closed_date",
    "content",
    "createdate",
    "data_creation_stage",
    "how_can_we_help_",
    "how_can_we_help___transit_agency_",
    "hs_file_upload",
    "hs_lastactivitydate",
    "hs_object_id",
    "hs_tag_ids",
    "hubspot_owner_id",
    "is_this_agency_remix_eligible_",
    "is_this_an_mtc_511_agency_",
    "issue_type",
    "last_reply_date",
    "latest_status",
    "n2024_holiday_outreach_campaign_final_status",
    "pads_validators",
    "proactive_reactive",
    "schedule_feed",
    "scheduling_campaign",
    "service_alerts_feed",
    "should_the_am_reach_out_to_encourage_remix_",
    "source_type",
    "support_ticket_subject",
    "team_supporting",
    "ticket_subject_matter",
    "time_to_close",
    "time_to_first_agent_reply",
    "trip_updates_feed",
    "vehicle_positions_feed",
    "what_is_the_caltrans_district_number_",
    "trip_planners_are_notified_by_Caltrans_of_change_in_GTFS_URL",
    "payments_team_is_notified_by_Caltrans_of_change_in_GTFS_URL",
    "other_stakeholders_identified_by_agency_are_notified_by_Caltrans_or_agency_of_change_in_GTFS_URL",
    "notes",
    "ntd_is_notified_by_caltrans_or_agency_re_gtfs_url_updates",
    "mtc_511_is_notified_of_change_in_GTFS_URL",
    "gtfsrt_provider_is_notified_by_caltrans_or_agency_re_updated_gtfs_url",
    "gtfs_aggregators_are_notified_by_Caltrans_of_change_in_GTFS_URL",
    "deployment_date_for_gtfs_to_be_live",
    "airtable_gtfs_dataset_records_created_for_new_links",
    "airtable_gtfs_service_data_records_set_to_customer_facing",
]
ticket_unused_props = [
    # There are none in the list above!
]

In [ ]:
# export summary of important properties to ticket_prop_summary.csv
get_object_prop_summary("ticket", ticket_wanted_props, ticket_unused_props)

## Deals

In [ ]:
# request properties and export to data/deal_props.json
if RUN_MODE == RunMode.FULL:
    get_object_props("deal")

In [ ]:
# define list of properties that we want to export
deal_wanted_props = [
    "dealname",
    "pipeline",
    "dealstage",
    "accepted_card_schemes",
    "hs_lastmodifieddate",
    "preferred_benefits_website_url__sync_",
    "active_benefits_flows",
    "additional_staff",
    "agency_logo__sync_",
    "agency_ntd_id",
    "agency_voms",
    "applicable_discount_types__sync_",
    "benefits_marketing_assistance__sync_",
    "benefits_testing_assistance__sync_",
    "board_approval__sync_",
    "campaign_status",
    "closed_lost_reason",
    "closedate",
    "createdate",
    "date_access_agreement_was_received",
    "dedicated_product_creation_person__sync_",
    "description_of_fare_caps__sync_",
    "desired_launch_timeline__sync_",
    "different_discount_fares__sync_",
    "discount_fare_caps",
    "test_sync",
    "does_agency_have_existing_gtfs_rt_data_",
    "does_the_agency_have_a_discounted_fare_program_",
    "engagements_last_meeting_booked",
    "existing_remix_customer_",
    "fare_type__sync_",
    "funding_source_s__used_for_payments_project",
    "help_with_product_creation__sync_",
    "hs_priority",
    "hs_tag_ids",
    "implementation",
    "in_person_verification__sync_",
    "is_agency_interested_in_exploring_gtfs_rt_contracts_",
    "is_the_agency_fare_free_",
    "latest_status",
    "next_step__digital_signage_pipeline_",
    "notes_last_contacted",
    "notes_last_updated",
    "notes_next_activity_date",
    "other_funding_sources",
    "pathways_to_be_enabled_for_benefits__sync_",
    "reason_benefits_is_n_a",
    "reason_why_participation_was_declined",
    "remix_agreement",
    "remix_research_outreach_status___round_one__onboarding_",
    "requested_pathways_to_enable",
    "requests_for_additional_support",
    "research_stage",
    "scheduling_software",
    "source_of_agency_interest",
    "stage_s__of_benefits_onboarding",
    "survey",
    "tap_agency",
    "targeted_launch_date",
    "used_msa_contracts",
    "voms__sync_",
    "which_digital_signage_vendors_responded_",
    "which_digital_signage_vendors_were_sent_proposal_",
    "workstream",
    "context_for_not_pursuing",
]
deal_unused_props = [
    "different_discount_fares__sync_",
    "research_stage",
    "workstream",
]

In [ ]:
# export summary of important properties to deal_prop_summary.csv
get_object_prop_summary("deal", deal_wanted_props, deal_unused_props)

## Vendors

In [ ]:
# request properties and export to data/vendor_props.json
if RUN_MODE == RunMode.FULL:
    get_object_props("2-22517187", nicename="vendor")

In [ ]:
# define list of properties that we want to export
vendor_wanted_props = [
    "vendor_name",
    "domain",
    "hs_created_by_user_id",
    "hs_createdate",
    "hs_lastmodifieddate",
    "hs_merged_object_ids",
    "hs_object_id",
    "hs_object_source_detail_1",
    "hs_object_source_detail_2",
    "hs_object_source_detail_3",
    "hs_object_source_label",
    "hs_pipeline",
    "hs_pipeline_stage",
    "hs_shared_team_ids",
    "hs_shared_user_ids",
    "hs_updated_by_user_id",
    "hubspot_owner_assigneddate",
    "hubspot_owner_id",
    "hubspot_team_id",
    "vendor_details__connectivity",
    "vendor_details__fare_collection",
    "vendor_details__integration",
    "vendor_details__location",
    "vendor_details__onboard_rider_comms",
    "vendor_details__operator_data",
    "vendor_details__regular_bus",
    "vendor_details__safety_tech",
    "vendor_details__transit_operations",
    "vendor_details__zev",
    "vendor_type",
]
vendor_unused_props = [
    # There are none in the list above!
]

In [ ]:
# export summary of important properties to vendor_prop_summary.csv
get_object_prop_summary("2-22517187", vendor_wanted_props, vendor_unused_props, nicename="vendor")

## Emails

In [ ]:
# request properties and export to data/email_props.json
# - HAD TO ADD `sales-email-read` SCOPE TO LEGACY APP
if RUN_MODE == RunMode.FULL:
    get_object_props("email")

In [ ]:
# define list of properties that we want to export
email_wanted_props = [
    "hs_object_id",
    "hs_all_accessible_team_ids",
    "hs_all_assigned_business_unit_ids",
    "hs_all_owner_ids",
    "hs_all_team_ids",
    "hs_app_id",
    "hs_at_mentioned_owner_ids",
    "hs_attachment_ids",
    "hs_body_preview",
    "hs_body_preview_html",
    "hs_body_preview_is_truncated",
    "hs_campaign_guid",
    "hs_campaign_guids",
    "hs_created_by",
    "hs_created_by_user_id",
    "hs_createdate",
    "hs_direction_and_unique_id",
    "hs_email_associated_contact_id",
    "hs_email_attached_video_id",
    "hs_email_attached_video_name",
    "hs_email_attached_video_opened",
    "hs_email_attached_video_watched",
    "hs_email_bcc_email",
    "hs_email_bcc_firstname",
    "hs_email_bcc_lastname",
    "hs_email_bcc_raw",
    "hs_email_bounce_error_detail_message",
    "hs_email_bounce_error_detail_status_code",
    "hs_email_cc_email",
    "hs_email_cc_firstname",
    "hs_email_cc_lastname",
    "hs_email_cc_raw",
    "hs_email_click_count",
    "hs_email_click_rate",
    "hs_email_details",
    "hs_email_direction",
    "hs_email_encoded_email_associations_request",
    "hs_email_error_message",
    "hs_email_facsimile_send_id",
    "hs_email_from_email",
    "hs_email_from_firstname",
    "hs_email_from_lastname",
    "hs_email_from_raw",
    "hs_email_has_inline_images_stripped",
    "hs_email_headers",
    "hs_email_html",
    "hs_email_logged_from",
    "hs_email_media_processing_status",
    "hs_email_member_of_forwarded_subthread",
    "hs_email_message_id",
    "hs_email_migrated_via_portal_data_migration",
    "hs_email_ms_teams_payload",
    "hs_email_open_count",
    "hs_email_open_rate",
    "hs_email_pending_inline_image_ids",
    "hs_email_post_send_status",
    "hs_email_recipient_drop_reasons",
    "hs_email_reply_count",
    "hs_email_reply_rate",
    "hs_email_send_event_id",
    "hs_email_send_event_id_created",
    "hs_email_sender_email",
    "hs_email_sender_firstname",
    "hs_email_sender_lastname",
    "hs_email_sender_raw",
    "hs_email_sent_count",
    "hs_email_sent_via",
    "hs_email_status",
    "hs_email_stripped_attachment_count",
    "hs_email_subject",
    "hs_email_text",
    "hs_email_thread_id",
    "hs_email_thread_summary",
    "hs_email_to_email",
    "hs_email_to_firstname",
    "hs_email_to_lastname",
    "hs_email_to_raw",
    "hs_email_tracker_key",
    "hs_email_validation_skipped",
    "hs_engagement_source",
    "hs_engagement_source_id",
    "hs_engagements_last_contacted",
    "hs_follow_up_action",
    "hs_gdpr_deleted",
    "hs_in_reply_to_engagement_id",
    "hs_incoming_email_is_out_of_office",
    "hs_lastmodifieddate",
    "hs_merged_object_ids",
    "hs_modified_by",
    "hs_not_tracking_opens_or_clicks",
    "hs_obj_coords",
    "hs_object_source",
    "hs_object_source_detail_1",
    "hs_object_source_detail_2",
    "hs_object_source_detail_3",
    "hs_object_source_id",
    "hs_object_source_label",
    "hs_object_source_user_id",
    "hs_owner_ids_bcc",
    "hs_owner_ids_cc",
    "hs_owner_ids_from",
    "hs_owner_ids_to",
    "hs_owning_teams",
    "hs_pinned_engagement_id",
    "hs_product_name",
    "hs_queue_membership_ids",
    "hs_read_only",
    "hs_scs_association_status",
    "hs_scs_audit_id",
    "hs_sequence_id",
    "hs_shared_team_ids",
    "hs_shared_user_ids",
    "hs_template_id",
    "hs_ticket_create_date",
    "hs_timestamp",
    "hs_unique_creation_key",
    "hs_unique_id",
    "hs_unique_tracker_key",
    "hs_updated_by_user_id",
    "hs_user_ids_of_all_notification_followers",
    "hs_user_ids_of_all_notification_unfollowers",
    "hs_user_ids_of_all_owners",
    "hs_was_imported",
    "hubspot_owner_assigneddate",
    "hubspot_owner_id",
    "hubspot_team_id",
]
email_unused_props = [
    "hs_all_assigned_business_unit_ids",
    "hs_at_mentioned_owner_ids",
    "hs_campaign_guid",
    "hs_campaign_guids",
    "hs_email_attached_video_id",
    "hs_email_attached_video_name",
    "hs_email_attached_video_opened",
    "hs_email_attached_video_watched",
    "hs_email_has_inline_images_stripped",
    "hs_email_migrated_via_portal_data_migration",
    "hs_email_ms_teams_payload",
    "hs_email_pending_inline_image_ids",
    "hs_email_send_event_id",
    "hs_email_send_event_id_created",
    "hs_email_sender_firstname",
    "hs_email_sender_lastname",
    "hs_email_sender_raw",
    "hs_email_stripped_attachment_count",
    "hs_email_thread_summary",
    "hs_email_validation_skipped",
    "hs_engagement_source_id",
    "hs_follow_up_action",
    "hs_in_reply_to_engagement_id",
    "hs_merged_object_ids",
    "hs_object_source_detail_1",
    "hs_object_source_detail_2",
    "hs_object_source_detail_3",
    "hs_pinned_engagement_id",
    "hs_product_name",
    "hs_queue_membership_ids",
    "hs_read_only",
    "hs_sequence_id",
    "hs_shared_team_ids",
    "hs_shared_user_ids",
    "hs_unique_creation_key",
    "hs_user_ids_of_all_notification_followers",
    "hs_user_ids_of_all_notification_unfollowers",
    "hs_was_imported",
]

In [ ]:
# export summary of important properties to email_prop_summary.csv
get_object_prop_summary("email", email_wanted_props, email_unused_props)

## Calls

In [ ]:
# request properties and export to data/call_props.json
if RUN_MODE == RunMode.FULL:
    get_object_props("call")

In [ ]:
# define list of properties that we want to export
call_wanted_props = [
    "hs_object_id",
    "hs_activity_type",
    "hs_all_accessible_team_ids",
    "hs_all_assigned_business_unit_ids",
    "hs_all_owner_ids",
    "hs_all_team_ids",
    "hs_at_mentioned_owner_ids",
    "hs_attachment_ids",
    "hs_body_preview",
    "hs_body_preview_html",
    "hs_body_preview_is_truncated", # none are truncated
    "hs_call_app_id",
    "hs_call_authed_url_provider",
    "hs_call_body",  # content with newlines
    "hs_call_callee_object_id",
    "hs_call_callee_object_type",
    "hs_call_deal_stage_during_call",
    "hs_call_direction",
    "hs_call_disposition",
    "hs_call_duration",
    "hs_call_external_account_id",
    "hs_call_external_id",
    "hs_call_from_number",
    "hs_call_from_number_nickname",
    "hs_call_has_transcript",
    "hs_call_has_voicemail",
    "hs_call_location",
    "hs_call_meeting_event_id",
    "hs_call_meeting_id",
    "hs_call_ms_teams_payload",
    "hs_call_phone_number_source",
    "hs_call_primary_deal",
    "hs_call_recording_duration",
    "hs_call_recording_url",
    "hs_call_routing_strategy_type",
    "hs_call_source",
    "hs_call_status",
    "hs_call_suggested_next_actions",
    "hs_call_summary",
    "hs_call_summary_execution_id",
    "hs_call_title",
    "hs_call_to_number",
    "hs_call_to_number_nickname",
    "hs_call_transcript_audio_num_channels",
    "hs_call_transcript_tracked_terms",
    "hs_call_transcription_id",
    "hs_call_unified_phone_number_id",
    "hs_call_unique_external_id",
    "hs_call_video_meeting_type",
    "hs_call_video_recording_url",
    "hs_call_zoom_meeting_uuid",
    "hs_calls_service_call_id",
    "hs_campaign_guid",
    "hs_campaign_guids",
    "hs_connected_count",
    "hs_created_by",
    "hs_created_by_user_id",
    "hs_createdate",
    "hs_engagement_source",
    "hs_engagement_source_id",
    "hs_engagements_last_contacted",
    "hs_follow_up_action",
    "hs_gdpr_deleted",
    "hs_hosted_video_conference_url",
    "hs_is_voice_agent_call",
    "hs_lastmodifieddate",
    "hs_merged_object_ids",
    "hs_modified_by",
    "hs_obj_coords",
    "hs_object_source",
    "hs_object_source_detail_1",
    "hs_object_source_detail_2",
    "hs_object_source_detail_3",
    "hs_object_source_id",
    "hs_object_source_label",
    "hs_object_source_user_id",
    "hs_owning_teams",
    "hs_pinned_engagement_id",
    "hs_product_name",
    "hs_queue_membership_ids",
    "hs_read_only",
    "hs_shared_team_ids",
    "hs_shared_user_ids",
    "hs_timestamp",
    "hs_unique_creation_key",
    "hs_unique_id",
    "hs_unknown_visitor_conversation",
    "hs_updated_by_user_id",
    "hs_user_ids_of_all_notification_followers",
    "hs_user_ids_of_all_notification_unfollowers",
    "hs_user_ids_of_all_owners",
    "hs_voice_agent_caller_requested_live_agent",
    "hs_voice_agent_escalated_to_live_agent",
    "hs_voice_agent_is_most_recent_customer_feedback_positive",
    "hs_voicemail_count",
    "hs_was_imported",
    "hubspot_owner_assigneddate",
    "hubspot_owner_id",
    "hubspot_team_id",
]
call_unused_props = [
    "hs_all_assigned_business_unit_ids",
    "hs_at_mentioned_owner_ids",
    "hs_attachment_ids",
    "hs_call_app_id",
    "hs_call_authed_url_provider",
    "hs_call_from_number_nickname",
    "hs_call_has_transcript",
    "hs_call_meeting_event_id",
    "hs_call_meeting_id",
    "hs_call_ms_teams_payload",
    "hs_call_recording_duration",
    "hs_call_recording_url",
    "hs_call_suggested_next_actions",
    "hs_call_summary",
    "hs_call_summary_execution_id",
    "hs_call_to_number_nickname",
    "hs_call_transcript_audio_num_channels",
    "hs_call_transcript_tracked_terms",
    "hs_call_transcription_id",
    "hs_call_unique_external_id",
    "hs_call_video_meeting_type",
    "hs_call_video_recording_url",
    "hs_call_zoom_meeting_uuid",
    "hs_campaign_guid",
    "hs_campaign_guids",
    "hs_follow_up_action",
    "hs_gdpr_deleted",
    "hs_hosted_video_conference_url",
    "hs_is_voice_agent_call",
    "hs_merged_object_ids",
    "hs_object_source_detail_1",
    "hs_object_source_detail_2",
    "hs_object_source_detail_3",
    "hs_pinned_engagement_id",
    "hs_product_name",
    "hs_queue_membership_ids",
    "hs_read_only",
    "hs_shared_team_ids",
    "hs_shared_user_ids",
    "hs_unique_creation_key",
    "hs_unknown_visitor_conversation",
    "hs_user_ids_of_all_notification_followers",
    "hs_user_ids_of_all_notification_unfollowers",
    "hs_voice_agent_caller_requested_live_agent",
    "hs_voice_agent_escalated_to_live_agent",
    "hs_voice_agent_is_most_recent_customer_feedback_positive",
    "hs_was_imported",
]

In [ ]:
# export summary of important properties to call_prop_summary.csv
get_object_prop_summary("call", call_wanted_props, call_unused_props)

## Meetings

In [ ]:
# request properties and export to data/meeting_props.json
if RUN_MODE == RunMode.FULL:
    get_object_props("meeting")

In [ ]:
# define list of properties that we want to export
meeting_wanted_props = [
    "hs_object_id",
    "hs_activity_type",
    "hs_all_accessible_team_ids",
    "hs_all_assigned_business_unit_ids",
    "hs_all_owner_ids",
    "hs_all_team_ids",
    "hs_associated_call_source",
    "hs_at_mentioned_owner_ids",
    "hs_attachment_ids",
    "hs_attendee_owner_ids",
    "hs_body_preview",
    "hs_body_preview_html",
    "hs_body_preview_is_truncated",
    "hs_booking_instance_id",
    "hs_campaign_guid",
    "hs_campaign_guids",
    "hs_contact_first_outreach_date",
    "hs_created_by",
    "hs_created_by_scheduling_page",
    "hs_created_by_user_id",
    "hs_createdate",
    "hs_customized_reminder_body_format",
    "hs_customized_reminder_subject_format",
    "hs_engagement_source",
    "hs_engagement_source_id",
    "hs_engagements_last_contacted",
    "hs_external_calendar_id",
    "hs_follow_up_action",
    "hs_follow_up_context",
    "hs_follow_up_tasks_remaining",
    "hs_gdpr_deleted",
    "hs_guest_emails",
    "hs_has_associated_notetaker_call_transcript",
    "hs_has_call_with_transcript",
    "hs_i_cal_uid",
    "hs_include_description_in_reminder",
    "hs_internal_meeting_notes",  # HTML, but there's no non-HTML version
    "hs_lastmodifieddate",
    "hs_meeting_body",  # some of these have newlines, some don't
    "hs_meeting_calendar_event_hash",
    "hs_meeting_change_id",
    "hs_meeting_created_from_link_id",
    "hs_meeting_end_time",
    "hs_meeting_external_url",
    "hs_meeting_location",
    "hs_meeting_location_type",
    "hs_meeting_ms_teams_payload",
    "hs_meeting_notetaker_id",
    "hs_meeting_notetaker_selection",
    "hs_meeting_outcome",
    "hs_meeting_payments_session_id",
    "hs_meeting_pre_meeting_prospect_reminders",
    "hs_meeting_source",
    "hs_meeting_source_id",
    "hs_meeting_start_time",
    "hs_meeting_title",
    "hs_meeting_web_conference_meeting_id",
    "hs_merged_object_ids",
    "hs_modified_by",
    "hs_notetaker_attendance_duration",
    "hs_obj_coords",
    "hs_object_source",
    "hs_object_source_detail_1",
    "hs_object_source_detail_2",
    "hs_object_source_detail_3",
    "hs_object_source_id",
    "hs_object_source_label",
    "hs_object_source_user_id",
    "hs_outcome_canceled_count",
    "hs_outcome_completed_count",
    "hs_outcome_no_show_count",
    "hs_outcome_rescheduled_count",
    "hs_outcome_scheduled_count",
    "hs_owning_teams",
    "hs_pinned_engagement_id",
    "hs_product_name",
    "hs_queue_membership_ids",
    "hs_read_only",
    "hs_recurring_event_schedule",
    "hs_recurring_event_series_id",
    "hs_recurring_event_type",
    "hs_recurring_event_unique_lookup",
    "hs_roster_object_coordinates",
    "hs_scheduled_tasks",
    "hs_shared_team_ids",
    "hs_shared_user_ids",
    "hs_time_to_book_meeting_from_first_contact",
    "hs_timestamp",
    "hs_timezone",
    "hs_unique_creation_key",
    "hs_unique_id",
    "hs_updated_by_user_id",
    "hs_user_ids_of_all_notification_followers",
    "hs_user_ids_of_all_notification_unfollowers",
    "hs_user_ids_of_all_owners",
    "hs_video_conference_platform",
    "hs_video_conference_url",
    "hs_was_imported",
    "hubspot_owner_assigneddate",
    "hubspot_owner_id",
    "hubspot_team_id",
]
meeting_unused_props = [
    "hs_associated_call_source",
    "hs_booking_instance_id",
    "hs_campaign_guid",
    "hs_campaign_guids",
    "hs_customized_reminder_body_format",
    "hs_customized_reminder_subject_format",
    "hs_external_calendar_id",
    "hs_follow_up_action",
    "hs_follow_up_context",
    "hs_guest_emails",
    "hs_has_associated_notetaker_call_transcript",
    "hs_meeting_ms_teams_payload",
    "hs_meeting_notetaker_id",
    "hs_meeting_notetaker_selection",
    "hs_meeting_payments_session_id",
    "hs_meeting_pre_meeting_prospect_reminders",
    "hs_merged_object_ids",
    "hs_notetaker_attendance_duration",
    "hs_object_source_detail_1",
    "hs_object_source_detail_2",
    "hs_object_source_detail_3",
    "hs_pinned_engagement_id",
    "hs_product_name",
    "hs_queue_membership_ids",
    "hs_read_only",
    "hs_roster_object_coordinates",
    "hs_shared_team_ids",
    "hs_shared_user_ids",
    "hs_unique_creation_key",
    "hs_user_ids_of_all_notification_followers",
    "hs_user_ids_of_all_notification_unfollowers",
]

In [ ]:
# export summary of important properties to meeting_prop_summary.csv
get_object_prop_summary("meeting", meeting_wanted_props, meeting_unused_props)

## Notes

In [ ]:
# request properties and export to data/note_props.json
if RUN_MODE == RunMode.FULL:
    get_object_props("note")

In [ ]:
# define list of properties that we want to export
note_wanted_props = [
    "hs_object_id",
    "hs_all_accessible_team_ids",
    "hs_all_assigned_business_unit_ids",
    "hs_all_owner_ids",
    "hs_all_team_ids",
    "hs_at_mentioned_owner_ids",
    "hs_attachment_from_form_submission",
    "hs_attachment_ids",
    "hs_body_preview",
    "hs_body_preview_html",
    "hs_body_preview_is_truncated",
    "hs_created_by",
    "hs_created_by_user_id",
    "hs_createdate",
    "hs_engagement_source",
    "hs_engagement_source_id",
    "hs_follow_up_action",
    "hs_gdpr_deleted",
    "hs_hd_ticket_ids",
    "hs_lastmodifieddate",
    "hs_merged_object_ids",
    "hs_modified_by",
    "hs_note_body",
    "hs_note_ms_teams_payload",
    "hs_obj_coords",
    "hs_object_source",
    "hs_object_source_detail_1",
    "hs_object_source_detail_2",
    "hs_object_source_detail_3",
    "hs_object_source_id",
    "hs_object_source_label",
    "hs_object_source_user_id",
    "hs_owning_teams",
    "hs_pinned_engagement_id",
    "hs_product_name",
    "hs_queue_membership_ids",
    "hs_read_only",
    "hs_shared_team_ids",
    "hs_shared_user_ids",
    "hs_timestamp",
    "hs_unique_creation_key",
    "hs_unique_id",
    "hs_updated_by_user_id",
    "hs_user_ids_of_all_notification_followers",
    "hs_user_ids_of_all_notification_unfollowers",
    "hs_user_ids_of_all_owners",
    "hs_was_imported",
    "hubspot_owner_assigneddate",
    "hubspot_owner_id",
    "hubspot_team_id",
]
note_unused_props = [
    "hs_all_assigned_business_unit_ids",
    "hs_attachment_from_form_submission",
    "hs_follow_up_action",
    "hs_merged_object_ids",
    "hs_note_ms_teams_payload",
    "hs_object_source_detail_2",
    "hs_object_source_detail_3",
    "hs_pinned_engagement_id",
    "hs_product_name",
    "hs_queue_membership_ids",
    "hs_read_only",
    "hs_shared_team_ids",
    "hs_shared_user_ids",
    "hs_unique_creation_key",
    "hs_unique_id",
    "hs_user_ids_of_all_notification_followers",
    "hs_user_ids_of_all_notification_unfollowers",
]

In [ ]:
# export summary of important properties to note_prop_summary.csv
get_object_prop_summary("note", note_wanted_props, note_unused_props)

## Tasks

In [ ]:
# request properties and export to data/task_props.json
if RUN_MODE == RunMode.FULL:
    get_object_props("task")

In [ ]:
# define list of properties that we want to export
task_wanted_props = [
    "hs_object_id",
    "hs_all_accessible_team_ids",
    "hs_all_assigned_business_unit_ids",
    "hs_all_owner_ids",
    "hs_all_team_ids",
    "hs_associated_company_labels",
    "hs_associated_contact_labels",
    "hs_at_mentioned_owner_ids",
    "hs_attachment_ids",
    "hs_body_preview",
    "hs_body_preview_html",
    "hs_body_preview_is_truncated",
    "hs_calendar_event_id",
    "hs_created_by",
    "hs_created_by_user_id",
    "hs_createdate",
    "hs_date_entered_60b5c368_04c4_4d32_9b4a_457e159f49b7_13292096",
    "hs_date_entered_61bafb31_e7fa_46ed_aaa9_1322438d6e67_1866552342",
    "hs_date_entered_af0e6a5c_2ea3_4c72_b69f_7c6cb3fdb591_1652950531",
    "hs_date_entered_dd5826e4_c976_4654_a527_b59ada542e52_2144133616",
    "hs_date_entered_fc8148fb_3a2d_4b59_834e_69b7859347cb_1813133675",
    "hs_date_exited_60b5c368_04c4_4d32_9b4a_457e159f49b7_13292096",
    "hs_date_exited_61bafb31_e7fa_46ed_aaa9_1322438d6e67_1866552342",
    "hs_date_exited_af0e6a5c_2ea3_4c72_b69f_7c6cb3fdb591_1652950531",
    "hs_date_exited_dd5826e4_c976_4654_a527_b59ada542e52_2144133616",
    "hs_date_exited_fc8148fb_3a2d_4b59_834e_69b7859347cb_1813133675",
    "hs_engagement_source",
    "hs_engagement_source_id",
    "hs_engagements_last_contacted",
    "hs_follow_up_action",
    "hs_gdpr_deleted",
    "hs_lastmodifieddate",
    "hs_marketing_studio_node_id",
    "hs_marketing_task_category",
    "hs_marketing_task_category_id",
    "hs_merged_object_ids",
    "hs_modified_by",
    "hs_msteams_message_id",
    "hs_obj_coords",
    "hs_object_id",
    "hs_object_source",
    "hs_object_source_detail_1",
    "hs_object_source_detail_2",
    "hs_object_source_detail_3",
    "hs_object_source_id",
    "hs_object_source_label",
    "hs_object_source_user_id",
    "hs_owning_teams",
    "hs_partner_map_category",
    "hs_partner_map_year",
    "hs_pinned_engagement_id",
    "hs_pipeline",
    "hs_pipeline_stage",
    "hs_product_name",
    "hs_queue_membership_ids",
    "hs_read_only",
    "hs_repeat_status",
    "hs_scheduled_tasks",
    "hs_shared_team_ids",
    "hs_shared_user_ids",
    "hs_start_date",
    "hs_start_date_with_fallback",
    "hs_target_duration",
    "hs_task_assigned_by_user_id",
    "hs_task_assigned_contacts",
    "hs_task_blocked_task_ids",
    "hs_task_blocking_task_ids",
    "hs_task_body",    # content with newlines
    "hs_task_campaign_guid",
    "hs_task_company_domain",
    "hs_task_company_industry",
    "hs_task_company_is_target_account",
    "hs_task_completion_count",
    "hs_task_completion_date",
    "hs_task_contact_email",
    "hs_task_contact_job_title",
    "hs_task_contact_phone",
    "hs_task_contact_timezone",
    "hs_task_family",
    "hs_task_for_object_type",
    "hs_task_is_all_day",
    "hs_task_is_completed",
    "hs_task_is_completed_call",
    "hs_task_is_completed_email",
    "hs_task_is_completed_linked_in",
    "hs_task_is_completed_sequence",
    "hs_task_is_open",
    "hs_task_is_overdue",
    "hs_task_is_past_due_date",
    "hs_task_is_sub_task",
    "hs_task_last_contact_outreach",
    "hs_task_last_sales_activity_timestamp",
    "hs_task_missed_due_date",
    "hs_task_missed_due_date_count",
    "hs_task_ms_teams_payload",
    "hs_task_parent_task_associated_objects",
    "hs_task_parent_task_id",
    "hs_task_parent_task_owner_ids",
    "hs_task_priority",
    "hs_task_probability_to_complete",
    "hs_task_relative_reminders",
    "hs_task_reminders",
    "hs_task_repeat_interval",
    "hs_task_send_default_reminder",
    "hs_task_sequence_enrollment_active",
    "hs_task_sequence_id",
    "hs_task_sequence_step_enrollment_contact_id",
    "hs_task_sequence_step_enrollment_id",
    "hs_task_sequence_step_number",
    "hs_task_sequence_step_order",
    "hs_task_status",
    "hs_task_sub_task_at_mentioned_owner_ids",
    "hs_task_sub_task_ids",
    "hs_task_sub_task_owner_ids",
    "hs_task_subject",
    "hs_task_template_id",
    "hs_task_type",
    "hs_task_uncompleted_sub_task_ids",
    "hs_time_in_60b5c368_04c4_4d32_9b4a_457e159f49b7_13292096",
    "hs_time_in_61bafb31_e7fa_46ed_aaa9_1322438d6e67_1866552342",
    "hs_time_in_af0e6a5c_2ea3_4c72_b69f_7c6cb3fdb591_1652950531",
    "hs_time_in_dd5826e4_c976_4654_a527_b59ada542e52_2144133616",
    "hs_time_in_fc8148fb_3a2d_4b59_834e_69b7859347cb_1813133675",
    "hs_timestamp",
    "hs_unique_creation_key",
    "hs_unique_id",
    "hs_universal_association_selector",
    "hs_updated_by_user_id",
    "hs_user_ids_of_all_notification_followers",
    "hs_user_ids_of_all_notification_unfollowers",
    "hs_user_ids_of_all_owners",
    "hs_was_imported",
    "hubspot_owner_assigneddate",
    "hubspot_owner_id",
    "hubspot_team_id",
]
task_unused_props = [
    "hs_all_assigned_business_unit_ids",
    "hs_attachment_ids",
    "hs_calendar_event_id",
    "hs_follow_up_action",
    "hs_marketing_studio_node_id",
    "hs_marketing_task_category",
    "hs_marketing_task_category_id",
    "hs_merged_object_ids",
    "hs_msteams_message_id",
    "hs_object_source_detail_2",
    "hs_object_source_detail_3",
    "hs_partner_map_category",
    "hs_partner_map_year",
    "hs_pinned_engagement_id",
    "hs_product_name",
    "hs_read_only",
    "hs_shared_team_ids",
    "hs_shared_user_ids",
    "hs_task_assigned_contacts",
    "hs_task_blocked_task_ids",
    "hs_task_blocking_task_ids",
    "hs_task_campaign_guid",
    "hs_task_company_is_target_account",
    "hs_task_ms_teams_payload",
    "hs_task_parent_task_associated_objects",
    "hs_task_parent_task_id",
    "hs_task_parent_task_owner_ids",
    "hs_task_sequence_enrollment_active",
    "hs_task_sequence_id",
    "hs_task_sequence_step_enrollment_contact_id",
    "hs_task_sequence_step_enrollment_id",
    "hs_task_sequence_step_number",
    "hs_task_sequence_step_order",
    "hs_task_sub_task_at_mentioned_owner_ids",
    "hs_task_sub_task_ids",
    "hs_task_sub_task_owner_ids",
    "hs_task_uncompleted_sub_task_ids",
    "hs_unique_creation_key",
    "hs_user_ids_of_all_notification_followers",
    "hs_user_ids_of_all_notification_unfollowers",
]

In [ ]:
# export summary of important properties to task_prop_summary.csv
get_object_prop_summary("task", task_wanted_props, task_unused_props)